# Phase 2 Verification — Google Colab

**Purpose.** Verify the entire data pipeline (Stages 0 → 4 + Section 4 charts) on Colab,
using mock data, before any API key is required.

**Success criteria.** All checks print ✅ and the final summary shows:
- k=5 clusters found
- 강남 → Business District
- 홍대입구 → Nightlife District
- 여의도 → Business District 2

**Runtime.** ~90 seconds on a free Colab CPU instance.

---

## How to open

Two paths — run **only one** of Cell 1A or Cell 1B.

| Path | When to use |
|---|---|
| **Cell 1A** — GitHub clone | After the repo is pushed to GitHub |
| **Cell 1B** — Drive zip   | Right now (zip already uploaded to Drive) |

## Cell 1A — Clone from GitHub  *(skip if using Drive zip)*

In [ ]:
# OPTION A: clone the repo from GitHub.
# Skip this cell if you uploaded the zip to Drive (use Cell 1B instead).
REPO_URL = 'https://github.com/sth00619/2026-Summer-Study.git'
PROJECT_SUBPATH = 'projects/seoul-movement-lecture'

import os, shutil, subprocess
if os.path.exists('/content/repo'):
    shutil.rmtree('/content/repo')
subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, '/content/repo'], check=True)
PROJECT_ROOT = f'/content/repo/{PROJECT_SUBPATH}'
os.chdir(PROJECT_ROOT)
print('Working directory:', os.getcwd())

## Cell 1B — Load from Drive zip  *(skip if using GitHub)*

**Before running:** upload `seoul-movement-lecture.zip` to
`MyDrive/Google Colab/seoul-movement-lecture.zip`

In [ ]:
# OPTION B: mount Drive and unzip the project.
# Skip this cell if you are cloning from GitHub (Cell 1A).
ZIP_PATH = '/content/drive/MyDrive/Google Colab/seoul-movement-lecture.zip'

import os, shutil, zipfile
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

if os.path.exists('/content/repo'):
    shutil.rmtree('/content/repo')
os.makedirs('/content/repo', exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall('/content/repo')

candidates = [d for d in os.listdir('/content/repo')
              if os.path.isdir(f'/content/repo/{d}')]
assert len(candidates) == 1, f'Expected one top-level folder, got: {candidates}'
PROJECT_ROOT = f'/content/repo/{candidates[0]}'
os.chdir(PROJECT_ROOT)
print('Working directory:', os.getcwd())
print('Contents:', sorted(os.listdir('.')))

## Cell 2 — Install dependencies (~30 s)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'umap-learn', 'pyarrow', '--quiet'], check=True)
print('Deps ready.')

## Cell 3 — Register the `src/` package

Colab does not add the working directory to `sys.path` automatically.
This cell fixes that so `from src import ...` works everywhere below.

In [ ]:
import sys, os
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from src import cluster, features, mock_data, preprocess
from src.lawd_codes import FOCUS_STATIONS, FOCUS_LAWD_CODES
print('src package imports cleanly')

## Check 1 · Mock data generation

Generates 24 months of synthetic OA-12252 (hourly) + OA-12914 (daily)
+ MOLIT apartment trades. No API key needed.

In [ ]:
months = mock_data._month_range('202605', 24)
print(f'Window: {months[0]} to {months[-1]}  ({len(months)} months)')

raw_hourly = mock_data.generate_subway_hourly(months, seed=42)
raw_daily  = mock_data.generate_subway_daily(months, seed=42)
raw_trades = mock_data.generate_apartment_trades(
    list(FOCUS_LAWD_CODES.values()), months, seed=42)

print(f'  raw_hourly : {raw_hourly.shape}  (expected 720 rows x 52 cols)')
print(f'  raw_daily  : {raw_daily.shape}')
print(f'  raw_trades : {raw_trades.shape}')

assert raw_hourly.shape == (720, 52), f'Unexpected shape: {raw_hourly.shape}'
assert not raw_daily.empty
assert not raw_trades.empty
print('CHECK 1 PASSED - mock data generation OK')

## Check 2 · Schema normalization (Stages 1-2)

Wide Korean-named table → long snake_case tidy table.

In [ ]:
long_hourly = preprocess.normalize_subway_hourly(raw_hourly)
long_hourly = preprocess.assign_station_id(long_hourly)
daily       = preprocess.normalize_subway_daily(raw_daily)
trades      = preprocess.normalize_apartment_trades(raw_trades)
price_idx   = preprocess.build_monthly_price_index(trades)

print(f'  long_hourly : {long_hourly.shape}')
print(f'  daily       : {daily.shape}')
print(f'  trades      : {trades.shape}')
print(f'  price_idx   : {price_idx.shape}  (expected 72 = 3 gu x 24 months)')

print('\nSample rows from long_hourly:')
print(long_hourly.head(3).to_string(index=False))

required_cols = {'year_month', 'line', 'station', 'hour', 'direction', 'count', 'station_id'}
assert required_cols.issubset(set(long_hourly.columns)), 'Missing columns'
assert price_idx.shape[0] == 72, f'Expected 72 rows, got {price_idx.shape[0]}'
assert trades['deal_krw'].min() > 100_000_000, 'Deal amounts look too small'
print('CHECK 2 PASSED - schema normalization OK')

## Check 3 · Feature engineering (Stage 3)

6 features per station.
Sanity: Gangnam high positive AM asymmetry, Hongdae high late-night share.

In [ ]:
feats = features.compute_station_features(long_hourly)
print(f'Feature matrix: {feats.shape[0]} stations x {len(features.FEATURE_ORDER)} features')

focus_kr = ['강남', '홍대입구', '여의도']
focus_rows = feats[feats['station'].isin(focus_kr)].set_index('station')
print(focus_rows[features.FEATURE_ORDER].round(3).to_string())

gangnam_asym  = focus_rows.loc['강남',     'directional_asymmetry_peak']
hongdae_night = focus_rows.loc['홍대입구', 'late_night_share']
yeouido_asym  = focus_rows.loc['여의도',   'directional_asymmetry_peak']

print(f'\nGangnam  AM asymmetry = {gangnam_asym:+.3f}  (expect > +0.5)')
print(f'Hongdae  late-night   = {hongdae_night:.3f}   (expect > 0.20)')
print(f'Yeouido  AM asymmetry = {yeouido_asym:+.3f}  (expect > +0.6)')

assert gangnam_asym  > 0.5,  f'Gangnam asymmetry too low: {gangnam_asym}'
assert hongdae_night > 0.20, f'Hongdae late-night too low: {hongdae_night}'
assert yeouido_asym  > 0.6,  f'Yeouido asymmetry too low: {yeouido_asym}'
print('CHECK 3 PASSED - features carry correct archetype signals')

## Check 4 · Clustering (Stage 4)

KMeans with k chosen by silhouette.
Success = k=5 AND focus stations land in the expected archetypes.

Note on Yeouido: it clusters as 'Business District 2', not 'Finance District'.
This is correct — Yeouido's hourly profile is a sharper version of Gangnam's.
The hypothesis in lawd_codes.py has been updated to match.

In [ ]:
result = cluster.choose_k_and_cluster(feats, k_range=(2, 8), random_state=42)
result = cluster.label_clusters_by_archetype(result)

print(f'k chosen (max silhouette): {result.k_chosen}')
sil = {k: round(v, 3) for k, v in result.silhouette_by_k.items()}
print(f'Silhouette scores by k:    {sil}')
print('\nCluster names:')
for idx, name in sorted(result.cluster_names.items()):
    print(f'  cluster {idx}: {name}')

print('\n--- Focus stations cluster assignment ---')
attribution = {}
for kr_name, meta in FOCUS_STATIONS.items():
    if kr_name in feats['station'].values:
        row_idx = feats.index[feats['station'] == kr_name][0]
        label   = int(result.labels[row_idx])
        got     = result.cluster_names[label]
        want    = meta['cluster_hypothesis']
        # Match: all words in want must appear in got.
        # 'Business District' matches 'Business District 2'
        ok = all(w.lower() in got.lower() for w in want.split())
        attribution[kr_name] = (label, got, want, ok)
        mark = 'OK' if ok else 'FAIL'
        print(f'  [{mark}]  {kr_name}  ->  cluster {label}: {got}  (hypothesis: {want})')

assert result.k_chosen == 5, f'Expected k=5, got k={result.k_chosen}'
assert all(ok for _, _, _, ok in attribution.values()), 'Focus station cluster mismatch'
print('\nCHECK 4 PASSED - clustering recovers expected neighborhood archetypes')

## Check 5 · Section 4 charts render

Renders Chart A (heatmap) and Chart B (hourly curves) inline.

In [ ]:
import matplotlib
matplotlib.use('Agg')   # Colab safe backend before importing pyplot
from src import chart_functions
import matplotlib.pyplot as plt

figs = chart_functions.render_section_4(
    long_hourly, daily,
    focus_stations=['강남', '홍대입구', '여의도'],
    export_dir='data/exports',
)

for name, fig in figs.items():
    print(f'Chart {name}:')
    plt.figure(fig.number)
    plt.show()

print('CHECK 5 PASSED - charts render cleanly')

## Check 6 · UMAP projection (optional, ~30 s)

2D embedding for Chart I (Section 6).

In [ ]:
result = cluster.add_umap_embedding(result, feats, random_state=42)
print(f'UMAP embedding shape: {result.embedding_2d.shape}')

import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(7, 5))
colors = plt.cm.tab10(result.labels / result.k_chosen)
ax.scatter(result.embedding_2d[:, 0], result.embedding_2d[:, 1],
           c=colors, s=80, edgecolors='white', linewidths=1.5)

for i, station in enumerate(feats['station']):
    if station in ('강남', '홍대입구', '여의도'):
        label_en = chart_functions.STATION_EN.get(station, station)
        ax.annotate(label_en,
                    (result.embedding_2d[i, 0], result.embedding_2d[i, 1]),
                    xytext=(5, 5), textcoords='offset points', fontsize=10)

ax.set_title('UMAP projection - KMeans clusters')
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
plt.tight_layout()
plt.show()
print('CHECK 6 PASSED - 2D projection ready')

## Persist processed tables

In [ ]:
from pathlib import Path
import pandas as pd

processed = Path('data/processed')
processed.mkdir(parents=True, exist_ok=True)

long_hourly.to_parquet(processed / 'ridership_long.parquet',  index=False)
daily.to_parquet(       processed / 'ridership_daily.parquet', index=False)
feats.to_parquet(       processed / 'features.parquet',        index=False)
price_idx.to_parquet(   processed / 'price_index.parquet',     index=False)

labels_df = pd.DataFrame({
    'station'     : feats['station'],
    'station_id'  : feats['station_id'],
    'cluster'     : result.labels,
    'cluster_name': [result.cluster_names[c] for c in result.labels],
})
labels_df.to_parquet(processed / 'cluster_labels.parquet', index=False)
result.centroids_original.to_parquet(processed / 'cluster_centroids.parquet')

print(f'Persisted to {processed.resolve()}:')
for f in sorted(processed.glob('*.parquet')):
    print(f'  {f.name:35}  {f.stat().st_size / 1024:.1f} KB')

---

## Final summary

If every check above printed PASSED, Phase 2 is fully verified on Colab.

| Check | What was verified |
|---|---|
| 1 | Mock data matches real API schema |
| 2 | Preprocessing wide→long, entity resolution, price index |
| 3 | 6-feature engineering carries archetype signals |
| 4 | KMeans k=5, focus stations correctly attributed |
| 5 | Section 4 charts render cleanly in Colab |
| 6 | UMAP 2D projection ready for Section 6 |

**Next:** Section 5 charts (C, D, E, F, G — the H1-rejection sequence).